<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [ ]:
import ee
import pandas as pd
import openpyxl
import ee
import datetime
import os
from PIL import Image
import numpy as np
import json
import IPython.display as disp
import csv

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project="ee-cima")
print(ee.String('Hello from the Earth Engine servers!').getInfo())

Hello from the Earth Engine servers!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/EEAPI/s1_ard.py"  "/content/"
! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/EEAPI/terrain_flattening.py"  "/content/"
! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/EEAPI/helper.py"  "/content/"
! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/EEAPI/speckle_filter.py"  "/content/"
! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/EEAPI/border_noise_correction.py"  "/content/"
! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/EEAPI/wrapper.py"  "/content/"
! cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/Apis/oefa_cde_img_result3.csv"  "/content/"
! cp "/content/drive/Othercomputers/MiPC/Code_OilSpill/Apis/oefa_cde_img_result3_fullvalues.xlsx"  "/content/"
! cp "/content/drive/Othercomputers/Mi PC/Code_OilSpill/Apis/last_dance/oefa_and_osig_ultimo.xlsx"  "/content/"


In [ ]:
df = pd.read_excel('/content/oefa_and_osig_ultimo.xlsx')


In [ ]:
df.columns

Index(['ADMINISTRADO', 'UNIDAD', 'FECHA DE EMERGENCIA AMBIENTAL', 'REGION',
       'TIPO', 'CANTIDAD_GALONES', 'DATOS', 'LAT', 'LON', 'FUENTE',
       'UBICACIÓN', 'CANTIDAD_FUENTE', 'OBS', 'datetime', 'coordinates',
       'Original index', 'geometry', 'bbox', 'bbox_converted'],
      dtype='object')

## Test the API

Coordenadas formato json **ee**


**SAVE FILES GOOGLE DRIVE**





In [ ]:
import sys
import os
import wrapper as wp

In [ ]:
from IPython.display import Image
import pandas as pd
from datetime import timedelta

In [ ]:
def data_str(df):
    id = df["Original index"]
    return id

In [ ]:
data_str(df)

,Original index
0,5
1,6
2,7
3,8
4,9
5,10
6,11
7,12


In [ ]:
invalid_rows = df[df['FECHA DE EMERGENCIA AMBIENTAL'].isna()]

# Print the problematic rows
if not invalid_rows.empty:
    print("Rows with invalid or missing dates:")
    print(invalid_rows[['FECHA DE EMERGENCIA AMBIENTAL', 'bbox_converted']])

# Iterate through rows and process valid dates
for index, row in df.iterrows():
    # Check if the date is valid
    if pd.isna(row['FECHA DE EMERGENCIA AMBIENTAL']):
        print(f"Skipping row {index} due to invalid date.")
        continue

    try:
        start_date = pd.to_datetime(row['FECHA DE EMERGENCIA AMBIENTAL'])
        end_date = start_date + timedelta(days=17)
        print(f"Start date: {start_date.strftime('%Y-%m-%d')}")
        print(f"End date: {end_date.strftime('%Y-%m-%d')}")

        # Load coordinates if they exist
        if isinstance(row['bbox_converted'], str):
            coordinates = json.loads(row['bbox_converted'])
            polygon = ee.Geometry.Polygon(coordinates)
            print(f"Polygon info: {polygon.getInfo()}")
        else:
            print(f"Invalid or missing bbox data in row {index}.")

    except Exception as e:
        print(f"Error processing row {index}: {e}")

Start date: 2014-06-01
End date: 2014-06-18
Polygon info: {'type': 'Polygon', 'coordinates': [[[-80.99715122556157, -3.8081546187945423], [-80.59715122556156, -3.8081546187945423], [-80.59715122556156, -3.408154618794542], [-80.99715122556157, -3.408154618794542], [-80.99715122556157, -3.8081546187945423]]]}
Start date: 2014-09-08
End date: 2014-09-25
Polygon info: {'type': 'Polygon', 'coordinates': [[[-81.62603958723433, -4.811053401247224], [-81.22603958723433, -4.811053401247224], [-81.22603958723433, -4.411053401247224], [-81.62603958723433, -4.411053401247224], [-81.62603958723433, -4.811053401247224]]]}
Start date: 2014-09-22
End date: 2014-10-09
Polygon info: {'type': 'Polygon', 'coordinates': [[[-80.96692440834417, -3.7860120424922132], [-80.56692440834416, -3.7860120424922132], [-80.56692440834416, -3.386012042492213], [-80.96692440834417, -3.386012042492213], [-80.96692440834417, -3.7860120424922132]]]}
Start date: 2015-03-14
End date: 2015-03-31
Polygon info: {'type': 'Polyg

In [ ]:
log_entries = []

# inside the image loop:
log_entries.append({
    "index": index,
    "image_num": i+1,
    "acquisition_date": date,
    "bands": ','.join(bands),
    "coverage_pct": coverage_pct
})

NameError: name 'i' is not defined

In [ ]:
log_entries

[{'index': 7,
  'image_num': 2,
  'acquisition_date': '2022-01-25',
  'bands': 'VV,VH,angle',
  'coverage_pct': 85.81}]

In [ ]:
import json
import ee
import pandas as pd
from datetime import timedelta

log_entries = []

for index, row in df.iterrows():
    # Convert date
    start_date = pd.to_datetime(row['FECHA DE EMERGENCIA AMBIENTAL'], errors='coerce')
    if pd.isna(start_date):
        print(f"Invalid date for row {index}, skipping.")
        continue

    end_date = start_date + timedelta(days=16)

    # Define the ROI geometry
    try:
        roi = ee.Geometry.Polygon(json.loads(row["bbox_converted"]))
    except Exception as e:
        print(f"Error converting bbox for row {index}: {e}")
        continue
    print(f"\n[Row {index}] --- {start_date.date()} to {end_date.date()} ---")

    # Get raw Sentinel-1 collection (unfiltered by polarization)
    s1_raw = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterDate(start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d'))
              .filterBounds(roi))

    total_count = s1_raw.size().getInfo()
    print(f"  Total images found: {total_count}")

    if total_count == 0:
        continue

    # Loop through all images found
    s1_list = s1_raw.toList(total_count)

    # inside the image loop:
    for i in range(total_count):
        img = ee.Image(s1_list.get(i))

        # Image metadata
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
        bands = img.bandNames().getInfo()
        system_index = img.get('system:index').getInfo()

        # Coverage calculation
        img_geom = img.geometry()
        try:
            roi_area = roi.area().getInfo()
            intersect = img_geom.intersection(roi, ee.ErrorMargin(1))
            intersect_area = intersect.area().getInfo()
            coverage_pct = round(100 * intersect_area / roi_area, 2)
        except Exception as e:
            coverage_pct = 'ERR'
            print(f"    Geometry error: {e}")

        print(f"    ➤ Image {i+1}/{total_count}")
        print(f"      ID           : {system_index}")
        print(f"      Date         : {date}")
        print(f"      Bands        : {bands}")
        print(f"      Coverage     : {coverage_pct}% of ROI")

        log_entries.append({
            "index": int(df["Original index"][index]),
            "image_num": i+1,
            "system_index": system_index,
            "acquisition_date": date,
            "bands": ','.join(bands),
            "coverage_pct": coverage_pct,
            'roi': ee.Geometry.Polygon(json.loads(row["bbox_converted"]))
        })



[Row 0] --- 2014-06-01 to 2014-06-17 ---
  Total images found: 0

[Row 1] --- 2014-09-08 to 2014-09-24 ---
  Total images found: 0

[Row 2] --- 2014-09-22 to 2014-10-08 ---
  Total images found: 0

[Row 3] --- 2015-03-14 to 2015-03-30 ---
  Total images found: 3
    ➤ Image 1/3
      ID           : S1A_IW_GRDH_1SSV_20150316T234352_20150316T234417_005063_0065C1_5F75
      Date         : 2015-03-16
      Bands        : ['VV', 'angle']
      Coverage     : 100.0% of ROI
    ➤ Image 2/3
      ID           : S1A_IW_GRDH_1SSV_20150325T110123_20150325T110148_005187_0068B7_144E
      Date         : 2015-03-25
      Bands        : ['VV', 'angle']
      Coverage     : 57.72% of ROI
    ➤ Image 3/3
      ID           : S1A_IW_GRDH_1SSV_20150325T110148_20150325T110224_005187_0068B7_4028
      Date         : 2015-03-25
      Bands        : ['VV', 'angle']
      Coverage     : 36.82% of ROI

[Row 4] --- 2017-07-11 to 2017-07-27 ---
  Total images found: 3
    ➤ Image 1/3
      ID           : S1B_IW

In [ ]:
log_entries

[{'index': 8,
  'image_num': 1,
  'system_index': 'S1A_IW_GRDH_1SSV_20150316T234352_20150316T234417_005063_0065C1_5F75',
  'acquisition_date': '2015-03-16',
  'bands': 'VV,angle',
  'coverage_pct': 100.0,
  'roi': ee.Geometry({
    "functionInvocationValue": {
      "functionName": "GeometryConstructors.Polygon",
      "arguments": {
        "coordinates": {
          "constantValue": [
            [
              [
                -81.22694221451678,
                -4.773598887022488
              ],
              [
                -81.22694221451678,
                -4.373598887022488
              ],
              [
                -81.62694221451679,
                -4.373598887022488
              ],
              [
                -81.62694221451679,
                -4.773598887022488
              ],
              [
                -81.22694221451678,
                -4.773598887022488
              ]
            ]
          ]
        },
        "evenOdd": {
          "constant

In [ ]:
import ee
import pandas as pd
from datetime import timedelta

# Assuming log_entries is already defined as given
for idx, entry in enumerate(log_entries):
    # Convert date from the 'acquisition_date' field
    start_date = pd.to_datetime(entry['acquisition_date'], errors='coerce')
    if pd.isna(start_date):
        print(f"Invalid date for entry at index {idx}, skipping.")
        continue

    end_date = start_date + timedelta(days=1)

    # Use the provided ROI directly from the dictionary
    try:
        roi = entry["roi"]  # Already an ee.Geometry
    except Exception as e:
        print(f"Error retrieving ROI for entry at index {idx}: {e}")
        continue

    bands_available = entry.get('bands', '')
    has_vv = 'VV' in bands_available
    has_vh = 'VH' in bands_available

    if has_vv and has_vh:
        polarization = 'VVVH'
    elif has_vv:
        polarization = 'VV'
    elif has_vh:
        polarization = 'VH'
    else:
        print(f"No valid polarization found for entry at index {idx}, skipping.")
        continue


    # Define the parameter dictionary
    parameter = {
        'START_DATE': start_date.strftime('%Y-%m-%d'),
        'STOP_DATE': end_date.strftime('%Y-%m-%d'),
        'POLARIZATION': polarization,  # Alternatively, you could split entry['bands'] if needed
        'ROI': roi,
        'APPLY_BORDER_NOISE_CORRECTION': False,
        'APPLY_SPECKLE_FILTERING': True,
        'SPECKLE_FILTER_FRAMEWORK': 'MONO',
        'SPECKLE_FILTER': 'LEE',
        'SPECKLE_FILTER_KERNEL_SIZE': 7,
        'SPECKLE_FILTER_NR_OF_IMAGES': 10,
        'APPLY_TERRAIN_FLATTENING': True,
        'DEM': ee.Image('USGS/SRTMGL1_003'),
        'TERRAIN_FLATTENING_MODEL': 'VOLUME',
        'TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER': 0,
        'FORMAT': 'DB',
        'CLIP_TO_ROI': True,
        'SAVE_ASSET': True
    }


    print(entry['index'])
    print(entry['image_num'])
    # Call the preprocessing function (uncomment and adapt as needed)
    wp.s1_preproc(parameter,entry['index'] ,entry['image_num'],entry["system_index"])

    print(f'Processed data for event on {start_date.strftime("%Y-%m-%d")}')

8
1
daedjo
Number of images in collection: 1
Mono-temporal speckle filtering is completed
Radiometric terrain normalization is completed
Exporting 8_1_2015-03-16_S1_GRD_VV to oefa_img_v7_tiff
Processed data for event on 2015-03-16
8
2
daedjo
Number of images in collection: 1
Mono-temporal speckle filtering is completed
Radiometric terrain normalization is completed
Exporting 8_2_2015-03-25_S1_GRD_VV to oefa_img_v7_tiff
Processed data for event on 2015-03-25
8
3
daedjo
Number of images in collection: 1
Mono-temporal speckle filtering is completed
Radiometric terrain normalization is completed
Exporting 8_3_2015-03-25_S1_GRD_VV to oefa_img_v7_tiff
Processed data for event on 2015-03-25
9
1
daedjo
Number of images in collection: 1
Mono-temporal speckle filtering is completed
Radiometric terrain normalization is completed
Exporting 9_1_2017-07-18_S1_GRD_VVVH to oefa_img_v7_tiff
Processed data for event on 2017-07-18
9
2
daedjo
Number of images in collection: 1
Mono-temporal speckle filteri

In [ ]:

import importlib
import wrapper
importlib.reload(wrapper)

import wrapper as wp

In [ ]:
import json
import ee
import pandas as pd
from datetime import timedelta


for index, row in df.iterrows():
    # Convert date
    start_date = pd.to_datetime(row['FECHA DE EMERGENCIA AMBIENTAL'], errors='coerce')
    if pd.isna(start_date):
        print(f"Invalid date for row {index}, skipping.")
        continue

    end_date = start_date + timedelta(days=16)

    # Define the ROI geometry
    try:
        roi = ee.Geometry.Polygon(json.loads(row["bbox_converted"]))
    except Exception as e:
        print(f"Error converting bbox for row {index}: {e}")
        continue

    # Define parameter dictionary
    parameter = {
        'START_DATE': start_date.strftime('%Y-%m-%d'),
        'STOP_DATE': end_date.strftime('%Y-%m-%d'),
        'POLARIZATION': 'VV',
        'ROI': roi,
        'APPLY_BORDER_NOISE_CORRECTION': False,
        'APPLY_SPECKLE_FILTERING': True,
        'SPECKLE_FILTER_FRAMEWORK': 'MONO',
        'SPECKLE_FILTER': 'LEE',
        'SPECKLE_FILTER_KERNEL_SIZE': 7,
        'SPECKLE_FILTER_NR_OF_IMAGES': 10,
        'APPLY_TERRAIN_FLATTENING': True,
        'DEM': ee.Image('USGS/SRTMGL1_003'),
        'TERRAIN_FLATTENING_MODEL': 'VOLUME',
        'TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER': 0,
        'FORMAT': 'DB',
        'CLIP_TO_ROI': True,
        'SAVE_ASSET': True
    }

    # Call the preprocessing function
    #wp.s1_preproc(parameter, df["Original index"][index])


    print(f'Processed data for event on {start_date.strftime("%Y-%m-%d")}')

[Index 0] Found 0 total Sentinel-1 images between 2014-06-01 00:00:00 and 2014-06-17 00:00:00.
Processed data for event on 2014-06-01
[Index 1] Found 0 total Sentinel-1 images between 2014-09-08 00:00:00 and 2014-09-24 00:00:00.
Processed data for event on 2014-09-08
[Index 2] Found 0 total Sentinel-1 images between 2014-09-22 00:00:00 and 2014-10-08 00:00:00.
Processed data for event on 2014-09-22
[Index 3] Found 3 total Sentinel-1 images between 2015-03-14 00:00:00 and 2015-03-30 00:00:00.
Processed data for event on 2015-03-14
[Index 4] Found 3 total Sentinel-1 images between 2017-07-11 00:00:00 and 2017-07-27 00:00:00.
Processed data for event on 2017-07-11
[Index 5] Found 1 total Sentinel-1 images between 2021-11-04 00:00:00 and 2021-11-20 00:00:00.
Processed data for event on 2021-11-04
[Index 6] Found 3 total Sentinel-1 images between 2021-11-05 00:00:00 and 2021-11-21 00:00:00.
Processed data for event on 2021-11-05
[Index 7] Found 2 total Sentinel-1 images between 2022-01-15 0

In [ ]:
log_entries = []
for index, row in df.iterrows():
    start_date = pd.to_datetime(row['Fecha']) - timedelta(days=2)
    end_date = start_date + timedelta(days=5)
    roi
    log_entries.append({
        'file_name': f'{start_date.strftime("%Y%m%d")}_S1_GRD',
        'location': f'{ee.Geometry.Rectangle(json.loads(row["bbox"]))}',
        'time': start_date.strftime('%Y-%m-%d')
    })
print(log_entries)

[{'file_name': '20041215_S1_GRD', 'location': 'ee.Geometry({\n  "functionInvocationValue": {\n    "functionName": "GeometryConstructors.Polygon",\n    "arguments": {\n      "coordinates": {\n        "constantValue": [\n          [\n            [\n              -81.3318335,\n              -4.519687200000001\n            ],\n            [\n              -81.3318335,\n              -4.6396872\n            ],\n            [\n              -81.2718335,\n              -4.6396872\n            ],\n            [\n              -81.2718335,\n              -4.519687200000001\n            ]\n          ]\n        ]\n      },\n      "evenOdd": {\n        "constantValue": true\n      }\n    }\n  }\n})', 'time': '2004-12-15'}, {'file_name': '20090824_S1_GRD', 'location': 'ee.Geometry({\n  "functionInvocationValue": {\n    "functionName": "GeometryConstructors.Polygon",\n    "arguments": {\n      "coordinates": {\n        "constantValue": [\n          [\n            [\n              -81.3318335,\n     

In [ ]:
len(log_entries)

69

In [ ]:

parameter = {'START_DATE': '2018-01-01',
            'STOP_DATE': '2018-02-01',
            'POLARIZATION': 'VV',
            'ORBIT' : 'BOTH',
            'PLATFORM_NUMBER' : 'A',
            'ORBIT_NUM': None,
            'ROI': ee.Geometry.Rectangle([-47.1634, -3.00071, -45.92746, -5.43836]),
            'APPLY_BORDER_NOISE_CORRECTION': False,
            'APPLY_SPECKLE_FILTERING': True,
            'SPECKLE_FILTER_FRAMEWORK':'MONO',
            'SPECKLE_FILTER': 'LEE',
            'SPECKLE_FILTER_KERNEL_SIZE': 7,
            'SPECKLE_FILTER_NR_OF_IMAGES':10,
            'APPLY_TERRAIN_FLATTENING': True,
            'DEM': ee.Image('USGS/SRTMGL1_003'),
            'TERRAIN_FLATTENING_MODEL': 'VOLUME',
            'TERRAIN_FLATTENING_ADDITIONAL_LAYOVER_SHADOW_BUFFER':0,
            'FORMAT': 'DB',
            'CLIP_TO_ROI': True,
            'SAVE_ASSET': True
            }

    log_entries.append({
        'file_name': f'{start_date.strftime("%Y%m%d")}_S1_GRD',
        'location': f'{row["bbox_converted"]}',
        'time': start_date.strftime('%Y-%m-%d')
    })

wp.s1_preproc(parameter)

Number of images in collection:  13
Mono-temporal speckle filtering is completed
Radiometric terrain normalization is completed
Exporting S1A_IW_GRDH_1SDV_20180107T084137_20180107T084202_020046_022281_87D2 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180107T084202_20180107T084227_020046_022281_1045 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180107T084227_20180107T084252_020046_022281_EA84 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180112T084946_20180112T085011_020119_0224E5_2A8F to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180112T085011_20180112T085036_020119_0224E5_EF72 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180119T084137_20180119T084202_020221_02280E_5674 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180119T084202_20180119T084227_020221_02280E_8D56 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180119T084227_20180119T084252_020221_02280E_3A38 to GEE_SAR_processed
Exporting S1A_IW_GRDH_1SDV_20180124T084946_20180124T085011_020294_022A74_263C to

In [ ]:
task.status()

NameError: name 'task' is not defined

In [ ]:
first_image = s1_processed.first()
first_image

In [ ]:
# Fetch and print the image metadata
metadata = first_image.getInfo()
print(metadata)

{'type': 'Image', 'bands': [{'id': 'VV', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'dimensions': [28724, 22174], 'origin': [0, 1], 'crs': 'EPSG:32723', 'crs_transform': [10, 0, 330129.7609730883, 0, -10, 9738702.70357257]}, {'id': 'angle', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [21, 10], 'crs': 'EPSG:32723', 'crs_transform': [-12553.2307824397, -4308.04279434518, 617369.086477932, 2730.286116309464, -20051.85848106444, 9684077.970874848]}], 'properties': {'SLC_Processing_software_name': 'Sentinel-1 IPF', 'sliceProductFlag': 'true', 'S1TBX_Calibration_vers': '6.0.4', 'orbitNumber_start': 20046, 'GRD_Post_Processing_start': 1515322414640, 'sliceNumber': 2, 'GRD_Post_Processing_facility_site': 'Airbus DS-Newport', 'instrument': 'Synthetic Aperture Radar', 'GRD_Post_Processing_facility_name': 'Copernicus S1 Core Ground Segment - UPA', 'resolution': 'H', 'SLC_Processing_facility_name': 'Copernicus S1 Core Ground Segment - UPA', 'GRD_Post_Pro

In [ ]:
# Print specific properties
acquisition_date = first_image.date().format().getInfo()
print("Acquisition Date:", acquisition_date)

# Example to fetch and print projection information
projection = first_image.select('VV').projection().getInfo()
print("Projection Information:", projection)

# Getting band information
band_info = first_image.bandNames().getInfo()
print("Band Names:", band_info)

Acquisition Date: 2018-01-07T08:41:37
Projection Information: {'type': 'Projection', 'crs': 'EPSG:32723', 'transform': [10, 0, 330129.7609730883, 0, -10, 9738702.70357257]}
Band Names: ['VV', 'angle']


In [ ]:
vv_band = first_image.select('VV')

In [ ]:
vv_band

In [ ]:
link = first_image.getDownloadURL({
    'scale': 352,
    'crs': 'EPSG:4326',
    'fileFormat': "GeoTIFF",
    'region': first_image.geometry().coordinates().getInfo()})
print(link)

EEException: Pixel grid dimensions (9699311x8154615) must be less than or equal to 32768.

In [ ]:
# Define a smaller region
smaller_region = ee.Geometry.Rectangle([-81.31, -4.63, -81.29, -4.61])

link = first_image.getDownloadURL({
    'scale': 10,
    'crs': 'EPSG:4326',
    'fileFormat': "GeoTIFF",
    'region': smaller_region.coordinates().getInfo()
})
print(link)

https://earthengine.googleapis.com/v1/projects/ee-cima/thumbnails/e8a13dba1b659f51705fc30f2f52a615-41f6cc96c6b6988f55dbcfe1c7aa6da0:getPixels


In [ ]:
link = first_image.getDownloadURL({
    'fileFormat': 'GeoTIFF',
    'region': first_image.geometry()})
print(link)

EEException: Total request size (1437868918 bytes) must be less than or equal to 50331648 bytes.

In [ ]:
manual_region = ee.Geometry.Rectangle([-81.3318335, -4.6396872, -81.2718335, -4.5196872])

In [ ]:
f_32 = first_image.toFloat()
task = ee.batch.Export.image.toDrive(image=f_32,
                                     description='elevation_near_lyon_france',
                                     scale=30,
                                     region=f_32.geometry(),
                                     folder='GEE_Folder',
                                     crs='EPSG:4326')
task.start()

In [ ]:
task.status()


{'state': 'COMPLETED',
 'description': 'elevation_near_lyon_france',
 'priority': 100,
 'creation_timestamp_ms': 1713747086415,
 'update_timestamp_ms': 1713747667333,
 'start_timestamp_ms': 1713747092456,
 'task_type': 'EXPORT_IMAGE',
 'destination_uris': ['https://drive.google.com/#folders/1gfzDvoYPs4fUUuvUNZkHdyon3GcrNplV'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 8475.474609375,
 'id': '6MVBJMXQFULVXZ63X5WZFT7H',
 'name': 'projects/ee-cima/operations/6MVBJMXQFULVXZ63X5WZFT7H'}

In [ ]:
link = f_32.getDownloadURL({
    'crs': 'EPSG:4326',
    'fileFormat': 'GeoTIFF',
    'region': f_32.geometry()})
print(link)

https://earthengine.googleapis.com/v1/projects/ee-cima/thumbnails/f508d3d7e7fd11211763c65522990e59-8e52e5d22e9ee2e506517beefcc01db8:getPixels
